# 🛡️ LexGuard — CUAD QA Span Extraction Training

This notebook fine-tunes **`nlpaueb/bert-base-uncased-contracts`** (a BERT model pre-trained on legal contract text) on the **CUAD v1 dataset** for **Question Answering (Span Extraction)**.

### What this model does:
Given a contract paragraph and a risk-category question (e.g., *"What is the uncapped liability clause?"*), the model:
1. **Extracts the exact risky text span** (the AI highlighter)
2. **Returns a confidence score** (start × end probability)
3. **Detects "no risk"** when the paragraph is safe (`is_impossible`)

### Prerequisites:
1. Go to **Runtime > Change runtime type** and set **Hardware accelerator** to **T4 GPU**
2. Upload `CUAD_v1.json` to the Colab files pane (or mount Google Drive)
3. Have your HuggingFace **Write** token ready

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
%%capture
!pip install transformers datasets accelerate evaluate
!pip install huggingface_hub

print("✅ All dependencies installed.")

In [ ]:
# ============================================================
# Cell 2: Load & Explore the CUAD Dataset
# ============================================================
import json
import os

# Option A: If you uploaded CUAD_v1.json directly to Colab
CUAD_PATH = "CUAD_v1.json"

# Option B: If using Google Drive, uncomment these lines:
# from google.colab import drive
# drive.mount('/content/drive')
# CUAD_PATH = "/content/drive/MyDrive/path/to/CUAD_v1.json"

with open(CUAD_PATH, "r") as f:
    cuad_raw = json.load(f)

print(f"CUAD version: {cuad_raw['version']}")
print(f"Number of contracts: {len(cuad_raw['data'])}")

# Count QA statistics
total_qas = 0
answerable = 0
impossible = 0
for article in cuad_raw['data']:
    for para in article['paragraphs']:
        for qa in para['qas']:
            total_qas += 1
            if qa['is_impossible']:
                impossible += 1
            else:
                answerable += 1

print(f"\nTotal QA pairs: {total_qas:,}")
print(f"  Answerable (has risky span): {answerable:,}")
print(f"  Impossible (no risk found):  {impossible:,}")

# Show a sample
sample_qa = cuad_raw['data'][0]['paragraphs'][0]['qas'][0]
print(f"\n📋 Sample question: {sample_qa['question'][:100]}...")
print(f"   Answer: {sample_qa['answers'][0]['text'][:80]}")
print(f"   is_impossible: {sample_qa['is_impossible']}")

In [ ]:
# ============================================================
# Cell 3: Filter to LexGuard Risk Categories
# ============================================================
# We focus on the 13 categories most relevant to contract risk assessment.
# Each category maps to a risk tier for LexGuard's final verdict.

LEXGUARD_CATEGORIES = {
    # 🔴 High Risk
    "Uncapped Liability": "HIGH",
    "Liquidated Damages": "HIGH",
    "Change Of Control": "HIGH",
    "Ip Ownership Assignment": "HIGH",
    "Termination For Convenience": "HIGH",
    # 🟡 Medium Risk
    "Cap On Liability": "MEDIUM",
    "Non-Compete": "MEDIUM",
    "Exclusivity": "MEDIUM",
    "Anti-Assignment": "MEDIUM",
    "No-Solicit Of Employees": "MEDIUM",
    # 🟢 Low Risk / Informational
    "Governing Law": "LOW",
    "Insurance": "LOW",
    "Audit Rights": "LOW",
}

# Normalize category names for matching against CUAD question IDs
# CUAD IDs look like: "CONTRACT_NAME__Category_Name"
def extract_category_from_id(qa_id: str) -> str:
    """Extract the category name from a CUAD QA ID."""
    return qa_id.split("__")[-1].replace("_", " ").strip()

# Build the normalized lookup set
category_lookup = {cat.lower(): cat for cat in LEXGUARD_CATEGORIES}

# Filter the dataset
filtered_data = []
category_counts = {}

for article in cuad_raw['data']:
    contract_title = article['title']
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            cat_name = extract_category_from_id(qa['id'])
            cat_lower = cat_name.lower()

            if cat_lower in category_lookup:
                matched_cat = category_lookup[cat_lower]
                risk_tier = LEXGUARD_CATEGORIES[matched_cat]

                filtered_data.append({
                    'id': qa['id'],
                    'question': qa['question'],
                    'context': context,
                    'answers': qa['answers'],
                    'is_impossible': qa['is_impossible'],
                    'category': matched_cat,
                    'risk_tier': risk_tier,
                    'contract': contract_title,
                })

                category_counts[matched_cat] = category_counts.get(matched_cat, 0) + 1

print(f"✅ Filtered to {len(filtered_data):,} QA pairs across {len(LEXGUARD_CATEGORIES)} risk categories\n")
print("Category breakdown:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    tier = LEXGUARD_CATEGORIES[cat]
    emoji = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}[tier]
    answerable_ct = sum(1 for d in filtered_data if d['category'] == cat and not d['is_impossible'])
    print(f"  {emoji} {cat:30s} → {count:4d} total ({answerable_ct} with spans)")

In [ ]:
# ============================================================
# Cell 4: Train/Validation Split & Tokenization
# ============================================================
# CRITICAL: We split by CONTRACT, not by QA pair, to avoid data leakage.

import random
from datasets import Dataset
from transformers import AutoTokenizer

random.seed(42)

# --- Split by contract ---
all_contracts = list(set(d['contract'] for d in filtered_data))
random.shuffle(all_contracts)
split_idx = int(0.8 * len(all_contracts))
train_contracts = set(all_contracts[:split_idx])
val_contracts = set(all_contracts[split_idx:])

train_data = [d for d in filtered_data if d['contract'] in train_contracts]
val_data = [d for d in filtered_data if d['contract'] in val_contracts]

print(f"Train: {len(train_data):,} QA pairs from {len(train_contracts)} contracts")
print(f"Val:   {len(val_data):,} QA pairs from {len(val_contracts)} contracts")

# --- Convert to HuggingFace Datasets ---
def to_squad_format(data_list):
    """Convert our filtered data into the format expected by HF's QA preprocessing."""
    records = {
        'id': [],
        'question': [],
        'context': [],
        'answers': [],
    }
    for d in data_list:
        records['id'].append(d['id'])
        records['question'].append(d['question'])
        records['context'].append(d['context'])
        if d['is_impossible'] or len(d['answers']) == 0:
            records['answers'].append({'text': [], 'answer_start': []})
        else:
            # Use the first answer span for training
            records['answers'].append({
                'text': [d['answers'][0]['text']],
                'answer_start': [d['answers'][0]['answer_start']],
            })
    return Dataset.from_dict(records)

train_dataset = to_squad_format(train_data)
val_dataset = to_squad_format(val_data)

print(f"\n✅ Datasets created: train={len(train_dataset)}, val={len(val_dataset)}")

# --- Load Tokenizer ---
MODEL_NAME = "nlpaueb/bert-base-uncased-contracts"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"\n✅ Tokenizer loaded: {MODEL_NAME}")

# --- Preprocessing with Sliding Window ---
MAX_LENGTH = 384      # Max tokens per window
DOC_STRIDE = 128      # Overlap between windows

def preprocess_training(examples):
    """Tokenize with sliding window and compute start/end positions for answer spans."""
    questions = [q.strip() for q in examples['question']]
    
    tokenized = tokenizer(
        questions,
        examples['context'],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    
    start_positions = []
    end_positions = []
    
    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        
        # If no answer (is_impossible), point to [CLS]
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue
        
        # Character-level start/end of the answer
        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])
        
        # Find which tokens belong to the context (not the question)
        sequence_ids = tokenized.sequence_ids(i)
        
        # Find context start and end token indices
        ctx_start = 0
        while ctx_start < len(sequence_ids) and sequence_ids[ctx_start] != 1:
            ctx_start += 1
        ctx_end = len(sequence_ids) - 1
        while ctx_end >= 0 and sequence_ids[ctx_end] != 1:
            ctx_end -= 1
        
        # Check if the answer is within this window
        if offsets[ctx_start][0] > end_char or offsets[ctx_end][1] < start_char:
            # Answer not in this window → point to [CLS]
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # Find the token positions for the answer
            token_start = ctx_start
            while token_start <= ctx_end and offsets[token_start][0] <= start_char:
                token_start += 1
            start_positions.append(token_start - 1)
            
            token_end = ctx_end
            while token_end >= ctx_start and offsets[token_end][1] >= end_char:
                token_end -= 1
            end_positions.append(token_end + 1)
    
    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

def preprocess_validation(examples):
    """Tokenize validation data, keeping offset mapping and example IDs for evaluation."""
    questions = [q.strip() for q in examples['question']]
    
    tokenized = tokenizer(
        questions,
        examples['context'],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []
    
    for i in range(len(tokenized["input_ids"])):
        sample_index = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_index])
        
        # Set offset_mapping to None for question tokens (only keep context offsets)
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]
    
    return tokenized

# Apply preprocessing
print("⏳ Tokenizing training data (this may take a few minutes)...")
tokenized_train = train_dataset.map(
    preprocess_training,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train",
)

print("⏳ Tokenizing validation data...")
tokenized_val = val_dataset.map(
    preprocess_validation,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing val",
)

print(f"\n✅ Tokenization complete!")
print(f"   Train features (windows): {len(tokenized_train):,}")
print(f"   Val features (windows):   {len(tokenized_val):,}")

In [ ]:
# ============================================================
# Cell 5: Load the Legal-BERT Model for Question Answering
# ============================================================
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Total parameters:     {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.1f}M)")
print(f"   Model size (approx):  ~{total_params * 4 / 1e6:.0f} MB")

In [ ]:
# ============================================================
# Cell 6: Train with HuggingFace Trainer
# ============================================================
from transformers import TrainingArguments, Trainer, DefaultDataCollator

# Training hyperparameters
training_args = TrainingArguments(
    output_dir="./lexguard_qa_checkpoints",
    
    # Training config
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Evaluation & logging
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # Performance
    fp16=True,  # Use mixed precision on T4
    dataloader_num_workers=2,
    
    # Misc
    seed=42,
    report_to="none",  # Disable wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val.remove_columns(["example_id", "offset_mapping"]),
    tokenizer=tokenizer,
    data_collator=DefaultDataCollator(),
)

print("🚀 Starting training...")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   Train features: {len(tokenized_train):,}")
print(f"   Expected time: ~30-45 minutes on T4 GPU\n")

train_result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Total steps: {train_result.global_step}")
print(f"   Final train loss: {train_result.training_loss:.4f}")

In [ ]:
# ============================================================
# Cell 7: Evaluate — Exact Match (EM) & F1 Score
# ============================================================
import collections
import numpy as np
import torch

def compute_qa_predictions(trainer, tokenized_val, val_dataset, n_best=20, max_answer_length=200):
    """
    Run the model on validation features and map predictions back to
    original QA examples using offset mappings.
    """
    # Get raw predictions
    raw_predictions = trainer.predict(tokenized_val)
    start_logits = raw_predictions.predictions[0]
    end_logits = raw_predictions.predictions[1]

    # Map features back to examples
    example_to_features = collections.defaultdict(list)
    for idx, example_id in enumerate(tokenized_val["example_id"]):
        example_to_features[example_id].append(idx)

    predictions = {}
    
    for example in val_dataset:
        example_id = example["id"]
        context = example["context"]
        
        best_answer = {"text": "", "score": 0.0}
        
        for feature_idx in example_to_features[example_id]:
            start_logit = start_logits[feature_idx]
            end_logit = end_logits[feature_idx]
            offsets = tokenized_val["offset_mapping"][feature_idx]
            
            # Get top start/end indices
            start_indices = np.argsort(start_logit)[-n_best:][::-1]
            end_indices = np.argsort(end_logit)[-n_best:][::-1]
            
            for start_idx in start_indices:
                for end_idx in end_indices:
                    # Skip invalid combinations
                    if start_idx >= len(offsets) or end_idx >= len(offsets):
                        continue
                    if offsets[start_idx] is None or offsets[end_idx] is None:
                        continue
                    if end_idx < start_idx:
                        continue
                    if end_idx - start_idx + 1 > max_answer_length:
                        continue
                    
                    score = start_logit[start_idx] + end_logit[end_idx]
                    if score > best_answer["score"]:
                        best_answer = {
                            "text": context[offsets[start_idx][0]:offsets[end_idx][1]],
                            "score": score,
                        }
        
        # Compare with null answer (CLS score)
        null_score = 0.0
        for feature_idx in example_to_features[example_id]:
            null_score = max(null_score, start_logits[feature_idx][0] + end_logits[feature_idx][0])
        
        # If null score is higher, predict empty (no risk)
        if null_score > best_answer["score"]:
            predictions[example_id] = ""
        else:
            predictions[example_id] = best_answer["text"]
    
    return predictions


def compute_f1(prediction, truth):
    """Token-level F1 between prediction and ground truth."""
    pred_tokens = prediction.lower().split()
    truth_tokens = truth.lower().split()
    
    if not truth_tokens and not pred_tokens:
        return 1.0
    if not truth_tokens or not pred_tokens:
        return 0.0
    
    common = collections.Counter(pred_tokens) & collections.Counter(truth_tokens)
    num_same = sum(common.values())
    
    if num_same == 0:
        return 0.0
    
    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_exact(prediction, truth):
    return int(prediction.strip().lower() == truth.strip().lower())


# Run evaluation
print("⏳ Running evaluation on validation set...")
predictions = compute_qa_predictions(trainer, tokenized_val, val_dataset)

# Compute metrics
f1_scores = []
em_scores = []

for example in val_dataset:
    example_id = example["id"]
    pred = predictions.get(example_id, "")
    
    truths = example["answers"]["text"]
    if not truths:
        # is_impossible — check if model correctly predicted empty
        f1_scores.append(1.0 if pred == "" else 0.0)
        em_scores.append(1 if pred == "" else 0)
    else:
        # Take max F1 across all valid answer spans
        best_f1 = max(compute_f1(pred, t) for t in truths)
        best_em = max(compute_exact(pred, t) for t in truths)
        f1_scores.append(best_f1)
        em_scores.append(best_em)

avg_f1 = np.mean(f1_scores) * 100
avg_em = np.mean(em_scores) * 100

print(f"\n" + "="*50)
print(f"📊 EVALUATION RESULTS")
print(f"="*50)
print(f"  Exact Match (EM): {avg_em:.1f}%")
print(f"  F1 Score:         {avg_f1:.1f}%")
print(f"  Evaluated on:     {len(f1_scores)} QA pairs")
print(f"="*50)

In [ ]:
# ============================================================
# Cell 8: Push Model to HuggingFace Hub
# ============================================================
from huggingface_hub import login

# ⚠️ REPLACE THESE WITH YOUR ACTUAL VALUES
HF_TOKEN = "hf_YOUR_WRITE_TOKEN_HERE"   # Your HuggingFace write token
HF_REPO = "your-username/LexGuard-CUAD-QA"  # Your repo name

login(token=HF_TOKEN)

# Save locally first
trainer.save_model("./lexguard_qa_final")
tokenizer.save_pretrained("./lexguard_qa_final")

# Push to Hub
print(f"\n⬆️ Pushing model to HuggingFace Hub: {HF_REPO}")
model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

print(f"\n✅ Model published to: https://huggingface.co/{HF_REPO}")
print(f"   You can now use it in your Streamlit app with:")
print(f'   pipeline("question-answering", model="{HF_REPO}")')

In [ ]:
# ============================================================
# Cell 9: Test Inference — Interactive Demo
# ============================================================
from transformers import pipeline

# Load the trained model as a QA pipeline
qa_pipeline = pipeline(
    "question-answering",
    model="./lexguard_qa_final",
    tokenizer="./lexguard_qa_final",
    device=0,  # GPU
)

# LexGuard risk questions (one per category)
RISK_QUESTIONS = {
    "Uncapped Liability": 'Highlight the parts (if any) of this contract related to "Uncapped Liability" that should be reviewed by a lawyer. Details: Is a party\'s liability uncapped upon the breach of its obligation in the contract?',
    "Liquidated Damages": 'Highlight the parts (if any) of this contract related to "Liquidated Damages" that should be reviewed by a lawyer. Details: Does the contract contain a clause that would award either party liquidated damages for breach or a fee upon the termination of a contract?',
    "Termination For Convenience": 'Highlight the parts (if any) of this contract related to "Termination For Convenience" that should be reviewed by a lawyer. Details: Can a party terminate this contract without cause?',
    "Cap On Liability": 'Highlight the parts (if any) of this contract related to "Cap On Liability" that should be reviewed by a lawyer. Details: Does the contract include a cap on liability upon the breach of a party\'s obligation?',
    "Non-Compete": 'Highlight the parts (if any) of this contract related to "Non-Compete" that should be reviewed by a lawyer. Details: Is there a restriction on the ability of a party to compete with the counterparty?',
}

# Test with a sample contract clause
test_context = """
SECTION 7.2 INDEMNIFICATION. The Licensee shall indemnify, defend, and hold harmless 
the Licensor from any and all claims, damages, losses, costs, and expenses (including 
reasonable attorneys' fees) arising out of or relating to the Licensee's use of the 
Licensed Materials, without any limitation of liability. The Licensee agrees that in 
the event of early termination by either party for convenience upon thirty (30) days 
written notice, a termination fee equal to the remaining contract value shall be due 
and payable immediately. Neither party shall engage in any business that directly 
competes with the other party's core business during the term of this Agreement and 
for a period of two (2) years following termination.
"""

print("=" * 60)
print("🛡️ LexGuard CUAD QA — Inference Demo")
print("=" * 60)
print(f"\n📄 Test clause ({len(test_context.split())} words):")
print(f"   {test_context.strip()[:200]}...\n")

RISK_MAP = {
    "Uncapped Liability": "HIGH",
    "Liquidated Damages": "HIGH",
    "Termination For Convenience": "HIGH",
    "Cap On Liability": "MEDIUM",
    "Non-Compete": "MEDIUM",
}

for category, question in RISK_QUESTIONS.items():
    result = qa_pipeline(question=question, context=test_context)
    tier = RISK_MAP[category]
    emoji = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}[tier]
    
    if result['score'] > 0.1 and result['answer'].strip():
        print(f"{emoji} {category} ({tier} RISK)")
        print(f"   Confidence: {result['score']:.1%}")
        print(f"   Span: \"{result['answer'][:120]}\"")
        print(f"   Position: chars {result['start']}–{result['end']}")
    else:
        print(f"✅ {category}: No risk detected (score: {result['score']:.3f})")
    print()

print("\n🎉 Demo complete! Your model is ready to use in the Streamlit app.")